# 01 — Data Loading & Exploratory Data Analysis (Phase 1)

Thin wrapper around `src/data/load.py`. Purpose:
1. Load and structurally validate the raw dataset.
2. Inspect target balance, missingness, and feature types.
3. Record observations that *inform* (but do not auto-trigger) later
   config decisions — e.g. whether `ModelConfig.class_imbalance_strategy`
   should be turned on in Phase 3.

No feature names or business assumptions are hardcoded anywhere in `src/`;
everything below is discovered from the data itself.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import get_default_config
from src.data.load import load_and_validate_data

config = get_default_config()
config.data.data_path = "data/raw/credit_risk_dataset.csv"  # adjust to actual file
config.data.target_column = "<set explicitly — dataset-specific>"

In [ ]:
df, schema = load_and_validate_data(config.data)
print(df.shape)
df.head()

In [ ]:
# Schema discovered from data, not hardcoded
print("Numerical features:", schema.numerical_features)
print("Categorical features:", schema.categorical_features)
print("Target column:", schema.target_column)

In [ ]:
df.info()
df.describe(include="all").T

## Missingness

In [ ]:
missing_ratio = df.isnull().mean().sort_values(ascending=False)
missing_ratio[missing_ratio > 0]

In [ ]:
flagged = missing_ratio[missing_ratio > config.data.missing_value_threshold]
print("Columns exceeding missing_value_threshold:", list(flagged.index))

## Duplicates

In [ ]:
if config.data.duplicate_check:
    n_dupes = df.duplicated().sum()
    print(f"Duplicate rows: {n_dupes}")

## Target distribution

This is the key input for deciding — *later, explicitly* — whether to enable
`ModelConfig.class_imbalance_strategy` in Phase 3. Observing imbalance here
does **not** auto-configure anything; it's a human decision recorded in
config, never inferred silently by `src/`.

In [ ]:
target = df[schema.target_column]
print(target.value_counts(normalize=True))

sns.countplot(x=target)
plt.title("Target class distribution")
plt.show()

## Numerical feature distributions & correlation

In [ ]:
df[schema.numerical_features].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[schema.numerical_features].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
plt.title("Numerical feature correlation")
plt.show()

## Categorical feature cardinality

In [ ]:
for col in schema.categorical_features:
    print(col, "->", df[col].nunique(), "unique values")
    print(df[col].value_counts(normalize=True).head())
    print()

## Observations log (fill in after running)

- Target imbalance ratio: `<fill in>`
- Columns with meaningful missingness: `<fill in>`
- High-cardinality categorical columns (watch for one-hot blow-up in Phase 2/7): `<fill in>`
- Any near-duplicate/leaky-looking features to flag for Phase 2: `<fill in>`